# M04 — Joins y KPIs

[← Anterior](../M03-transformacion-datos/03-lab-reglas-negocio.ipynb) · [Siguiente →](02-lab-joins.ipynb)

Un número de negocio mentiroso casi siempre viene de **cruzar mal** dos tablas, no de un `sum` mal escrito.

Vamos a montar un ejemplo mínimo: dos clientes reales y tres líneas. Una de las líneas apunta a un cliente que **no existe** (`CX9`). Eso en NovaShop son los huérfanos `CX*` que dejamos en el staging a propósito.

Ejecuta las celdas **aquí**, en este mismo fichero. No lo copies a otro sitio.

Kernel: **Python (NovaShop)**.


## Arranque

La primera celda **no es Spark todavía**: busca la raíz del repo (aunque este notebook no esté en la carpeta de arriba) y deja `RAW`, `STAGING` y `CURATED` listos. La segunda pide una `SparkSession` en `local[*]` (todos los cores de esta máquina; no hay clúster).

Al ejecutar: rutas impresas y una versión `3.5.x` con master `local[*]`.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
# getOrCreate: si ya hay sesión en este kernel, la reusa (mismo puerto 4040)
spark = get_spark('novashop-clase-m04')
print(spark.version, spark.sparkContext.master)


## Qué significa cada cruce

Un join responde: “para cada fila de la izquierda, ¿encuentro clave en la derecha?”.

- **inner**: solo las filas que **empatan**. `CX9` desaparece. El GMV de esa línea **no entra** en el total. Úsalo cuando el universo de negocio es “con cliente conocido”.
- **left**: todas las de la izquierda. `CX9` se queda, las columnas del cliente salen nulas. Úsalo para **medir** cuánto se pierde, no para reportar venta atribuida.
- **left_anti**: “está en la izquierda y **en ninguna** de la derecha”. Es la lista de huérfanos. Mejor que un `where` a ciegas.

Al ejecutar: inner **2**, left **3**, y el anti enseña la fila `O3` / `CX9`.


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col, sum as fsum, countDistinct

clientes = spark.createDataFrame([
    Row(customer_id="C1", country="ES"),
    Row(customer_id="C2", country="FR"),
])
lineas = spark.createDataFrame([
    Row(order_id="O1", customer_id="C1", gmv_line=100.0, is_billable=True),
    Row(order_id="O2", customer_id="C1", gmv_line=50.0, is_billable=True),
    Row(order_id="O3", customer_id="CX9", gmv_line=999.0, is_billable=True),  # huérfano
])
print("inner (pierde al huérfano)", lineas.join(clientes, "customer_id", "inner").count())
print("left  (conserva las 3)   ", lineas.join(clientes, "customer_id", "left").count())
print("quién no está en clientes:")
lineas.join(clientes, "customer_id", "left_anti").show()


## Ticket medio: grano pedido, no grano línea

`O1` y `O2` son **dos** pedidos del mismo cliente, 100 + 50 = 150. El ticket medio de la compañía en este juguete es `150 / 2 = 75`, no la media de las dos líneas (sigue siendo 75 aquí porque hay una línea por pedido; en NovaShop un pedido tiene varias líneas y `avg(gmv_line)` **baja** el ticket).

Regla: `sum(GMV) / countDistinct(order_id)`, siempre sobre el universo que hayas elegido (aquí: inner + cobrable). El `groupBy("country")` es el mismo GMV troceado.


In [ ]:
# Universo de dinero: cliente real y línea cobrable (el 999 de CX9 no entra)
sales = lineas.join(clientes, "customer_id", "inner").where(col("is_billable"))
sales.agg(
    fsum("gmv_line").alias("gmv"),
    countDistinct("order_id").alias("orders"),
).show()  # 150 y 2
sales.groupBy("country").agg(fsum("gmv_line").alias("gmv")).show()


**Siguiente:** [lab de joins](02-lab-joins.ipynb) sobre el dataset real.
